In [2]:
import requests
import pandas as pd
import pytz

def converter_data_trello(data_str):
    if not data_str or pd.isna(data_str):
        return pd.NaT
    dt_utc = pd.to_datetime(data_str, utc=True, errors='coerce')
    if pd.isna(dt_utc):
        return pd.NaT
    return dt_utc.tz_convert('America/Sao_Paulo')

NOME_QUADRO = "CTI "

AUTH = {
    "key": "94e7ea17c0af203c3719162f3a019b8b",
    "token": "ATTA64a25b844de4470670bb44db434a1cd9285d4186778dedafe131f041aa8dcd31CFB2E264"
}

def trello_get(endpoint, params=None):
    url = f"https://api.trello.com/1/{endpoint}"
    response = requests.get(url, params={**AUTH, **(params or {})})
    response.raise_for_status()
    return response.json()

# 1. Buscar quadro
boards = trello_get("members/me/boards")
board = next((b for b in boards if b['name'] == NOME_QUADRO), None)
print(f"✅ Quadro: {board['name']}")

# 2. Buscar listas (mapeamento ID -> Nome)
lists = trello_get(f"boards/{board['id']}/lists")
print("\n📋 LISTAS DO QUADRO:")
for lista in lists:
    print(f"  - '{lista['name']}'")
id_para_nome_lista = {lista['id']: lista['name'] for lista in lists}

# 3. Buscar cards
cards = trello_get(f"boards/{board['id']}/cards")
print(f"✅ {len(cards)} cards encontrados")

# 4. Buscar ações (apenas para data de criação)
all_actions = trello_get(f"boards/{board['id']}/actions", params={"filter": "createCard"})
print(f"✅ {len(all_actions)} ações de criação encontradas")

# Criar mapa de criação por card_id
card_criacao = {}
for action in all_actions:
    if action['type'] == 'createCard':
        card_id = action['data']['card']['id']
        card_criacao[card_id] = converter_data_trello(action['date'])

# 5. Processar cards
lead_times_list = []

for card in cards:
    card_id = card['id']
    card_name = card['name']
    
    # Data de criação (prioriza ação createCard)
    data_criacao = card_criacao.get(card_id)
    if pd.isna(data_criacao):
        data_criacao = converter_data_trello(card.get('dateLastActivity'))
    
    data_ultima_atividade = converter_data_trello(card.get('dateLastActivity'))
    
    # VERIFICAR LISTA ATUAL DO CARD
    lista_atual = id_para_nome_lista.get(card.get('idList'), '')
    
    # CONSIDERA CONCLUÍDO APENAS SE ESTIVER NA LISTA "Concluído"
    if lista_atual == 'Concluido':  # <--- NOME EXATO DA SUA LISTA
        status = 'Concluído'
        # Para cards já concluídos, usa a última atividade como data de conclusão
        data_conclusao = data_ultima_atividade
        lead_time_dias = (data_conclusao - data_criacao).days if pd.notna(data_criacao) and pd.notna(data_conclusao) else None
    else:
        status = 'Não concluido'
        data_conclusao = pd.NaT
        lead_time_dias = None
    
    lead_times_list.append({
        'card_id': card_id,
        'card_name': card_name,
        'status': status,
        'lista_atual': lista_atual,  # <-- adicionado para debug
        'data_criacao': data_criacao,
        'data_ultima_atividade': data_ultima_atividade,
        'data_conclusao': data_conclusao,
        'lead_time_dias': lead_time_dias,
    })

# 6. DataFrame
df_lead = pd.DataFrame(lead_times_list)

# 7. Exibir
print("\n📊 RESULTADO:")
display_df = df_lead.copy()
colunas_data = ['data_criacao', 'data_ultima_atividade', 'data_conclusao']
for col in colunas_data:
    if col in display_df.columns:
        display_df[col] = display_df[col].dt.strftime('%d/%m/%Y %H:%M:%S')

print(display_df[['card_name', 'lista_atual', 'status', 'lead_time_dias']].to_string())

# 8. Exportar
df_lead.to_csv('lead_time.csv', index=False, sep=';', date_format='%Y-%m-%d %H:%M:%S')
print("\n✅ lead_time.csv gerado")

✅ Quadro: CTI 

📋 LISTAS DO QUADRO:
  - 'Informações'
  - 'Datas'
  - 'A Fazer'
  - 'Sprint Atual'
  - 'Em Andamento'
  - 'Em Atraso'
  - 'Concluido'
✅ 20 cards encontrados
✅ 20 ações de criação encontradas

📊 RESULTADO:
                       card_name  lista_atual         status  lead_time_dias
0             Consultor: Janaina  Informações  Não concluido             NaN
1          Documento de abertura  Informações  Não concluido             NaN
2         Analista: Tiago Madrid  Informações  Não concluido             NaN
3         Documento de conclusão  Informações  Não concluido             NaN
4                    Alinhamento        Datas  Não concluido             NaN
5                       Kick-Off        Datas  Não concluido             NaN
6                   Configuração        Datas  Não concluido             NaN
7                   Double Check        Datas  Não concluido             NaN
8         Encaminhar dispositivo        Datas  Não concluido             NaN
9        

In [4]:
import requests
import pandas as pd
import pytz

def converter_data_trello(data_str):
    if not data_str or pd.isna(data_str):
        return pd.NaT
    dt_utc = pd.to_datetime(data_str, utc=True, errors='coerce')
    if pd.isna(dt_utc):
        return pd.NaT
    return dt_utc.tz_convert('America/Sao_Paulo')

NOME_QUADRO = "CTI "

AUTH = {
    "key": "94e7ea17c0af203c3719162f3a019b8b",
    "token": "ATTA64a25b844de4470670bb44db434a1cd9285d4186778dedafe131f041aa8dcd31CFB2E264"
}

def trello_get(endpoint, params=None):
    url = f"https://api.trello.com/1/{endpoint}"
    response = requests.get(url, params={**AUTH, **(params or {})})
    response.raise_for_status()
    return response.json()

# 1. Buscar quadro
boards = trello_get("members/me/boards")
board = next((b for b in boards if b['name'] == NOME_QUADRO), None)
if not board:
    raise Exception(f"Quadro '{NOME_QUADRO}' não encontrado")

print(f"✅ Conectado: {board['name']} (ID: {board['id']})")

# 2. Buscar listas
lists = trello_get(f"boards/{board['id']}/lists")
print("\n📋 LISTAS DO QUADRO:")
for lista in lists:
    print(f"  - '{lista['name']}'")
id_para_nome_lista = {lista['id']: lista['name'] for lista in lists}

# 3. Buscar cards
cards = trello_get(f"boards/{board['id']}/cards")
print(f"✅ {len(cards)} cards encontrados")

# 4. Buscar ações de criação
all_actions = trello_get(f"boards/{board['id']}/actions", params={"filter": "createCard"})
print(f"✅ {len(all_actions)} ações de criação encontradas")

# Mapa de criação
card_criacao = {}
for action in all_actions:
    if action['type'] == 'createCard':
        card_id = action['data']['card']['id']
        card_criacao[card_id] = converter_data_trello(action['date'])

# 5. Processar cards
lead_times_list = []

for card in cards:
    card_id = card['id']
    card_name = card['name']
    
    # Data de criação
    data_criacao = card_criacao.get(card_id)
    if pd.isna(data_criacao):
        data_criacao = converter_data_trello(card.get('dateLastActivity'))
    
    data_ultima_atividade = converter_data_trello(card.get('dateLastActivity'))
    
    # Verificar lista atual
    lista_atual = id_para_nome_lista.get(card.get('idList'), '')
    
    # Verificar labels (opcional)
    labels = card.get('labels', [])
    label_names = [label.get('name', '').lower() for label in labels]
    tem_label_concluido = any('conclu' in nome or 'done' in nome for nome in label_names)
    
    # CONSIDERA CONCLUÍDO SE ESTIVER NA LISTA "Concluido" (sem acento)
    if lista_atual == 'Concluido' or tem_label_concluido:
        status = 'Concluído'
        data_conclusao = data_ultima_atividade
        lead_time_dias = (data_conclusao - data_criacao).days if pd.notna(data_criacao) and pd.notna(data_conclusao) else None
    else:
        status = 'Não concluído'
        data_conclusao = pd.NaT
        lead_time_dias = None
    
    lead_times_list.append({
        'card_id': card_id,
        'card_name': card_name,
        'status': status,
        'lista_atual': lista_atual,
        'data_criacao': data_criacao,
        'data_ultima_atividade': data_ultima_atividade,
        'data_conclusao': data_conclusao,
        'lead_time_dias': lead_time_dias,
    })

# 6. DataFrame
df_lead = pd.DataFrame(lead_times_list)

# 7. Exibir
print("\n📊 RESULTADO:")
display_df = df_lead.copy()
colunas_data = ['data_criacao', 'data_ultima_atividade', 'data_conclusao']
for col in colunas_data:
    if col in display_df.columns:
        display_df[col] = display_df[col].dt.strftime('%d/%m/%Y %H:%M:%S')

print(display_df[['card_name', 'lista_atual', 'status', 'lead_time_dias']].to_string())

# 8. Estatísticas
concluidos = df_lead[df_lead['status'] == 'Concluído']
print(f"\n📊 Cards concluídos: {len(concluidos)}")
print(f"📊 Cards não concluídos: {len(df_lead) - len(concluidos)}")

if len(concluidos) > 0:
    print(f"⏱️ Lead time médio (dias): {concluidos['lead_time_dias'].mean():.2f}")
    print(f"⏱️ Lead time mínimo: {concluidos['lead_time_dias'].min()} dias")
    print(f"⏱️ Lead time máximo: {concluidos['lead_time_dias'].max()} dias")

# 9. Exportar CSV
df_lead.to_csv('lead_time.csv', index=False, sep=';', date_format='%Y-%m-%d %H:%M:%S')
print("\n✅ lead_time.csv gerado")

✅ Conectado: CTI  (ID: 69fd20465fb2b7d1fb85e050)

📋 LISTAS DO QUADRO:
  - 'Informações'
  - 'Datas'
  - 'A Fazer'
  - 'Sprint Atual'
  - 'Em Andamento'
  - 'Em Atraso'
  - 'Concluido'
✅ 20 cards encontrados
✅ 20 ações de criação encontradas

📊 RESULTADO:
                       card_name lista_atual     status  lead_time_dias
0                    Alinhamento   Concluido  Concluído               0
1                       Kick-Off   Concluido  Concluído               0
2                   Configuração   Concluido  Concluído               0
3                   Double Check   Concluido  Concluído               0
4         Encaminhar dispositivo   Concluido  Concluído               0
5                  Implementação   Concluido  Concluído               0
6         Reunião de finalização   Concluido  Concluído               0
7   Teste adicao de card - 11-05   Concluido  Concluído               3
8                        ticket:   Concluido  Concluído               6
9                    Alin

In [ ]:
# **Documentação da ETL – Projeto Integrador CTI**

*A ETL desenvolvida para o projeto integrador em parceria com CTI Brasil tem como objetivo coletar, tratar e organizar dados do quadro de tarefas da empresa na plataforma Trello, permitindo posteriormente a criação de dashboards para análise do fluxo operacional.*

*O processo foi implementado em Python e utiliza a API do Trello para acessar as informações do quadro da empresa. Inicialmente, o script realiza a autenticação por meio de chave e token da API, estabelecendo conexão com a conta responsável pelo quadro. Após essa conexão, o sistema localiza automaticamente o board utilizado pela empresa e extrai todos os cards cadastrados, além do histórico de ações relacionadas às movimentações das tarefas.*

*Na etapa de extração, são coletados dados como nome dos cards, identificadores, datas de atividade e histórico de mudanças entre listas. Para otimizar o desempenho, a ETL realiza apenas uma requisição para obter todos os cards do quadro e outra para obter todas as ações do tipo updateCard, evitando múltiplas chamadas individuais para cada tarefa.*

*Na etapa de transformação, os dados passam por tratamento com a biblioteca Pandas. As datas retornadas pela API, originalmente em UTC, são convertidas para o horário de Brasília, garantindo maior consistência nas análises. Em seguida, as informações são organizadas em estruturas tabulares, permitindo identificar a movimentação dos cards entre etapas do processo, como mudança de status entre listas do Trello.*

*Além disso, a ETL realiza o cálculo do lead time das tarefas, que representa o tempo decorrido entre a criação da atividade e sua conclusão. Para isso, o script verifica se o card foi movido para a lista “Concluído” e calcula a diferença entre as datas registradas. Caso a tarefa ainda não tenha sido finalizada, ela é classificada como “Não concluída”.*

*Ao final do processamento, são gerados três DataFrames principais: um contendo os dados gerais dos cards, outro com o histórico de ações e movimentações, e um terceiro com os indicadores de lead time. Esses dados podem ser exportados em formato CSV e utilizados como base para construção de dashboards analíticos.*

*A solução permite visualizar informações estratégicas sobre o fluxo de trabalho da empresa, como quantidade de tarefas, status das atividades, tempo médio de conclusão e histórico de movimentações. Dessa forma, o projeto transforma dados operacionais do Trello em informações estruturadas para análise e apoio à tomada de decisão no contexto do projeto integrador.*
